**Research Objective**: Fine-tuning Gemma-3-270M to act as a specialized decomposition agent for multi-hop queries, distilling logical hop-transition capabilities from GPT-4o.

Model choice (temporary):
This notebook uses `gemma-3-270m-it` to match the Unsloth tutorial. A controlled replication with `gemma-3-270m` (base) will be run to isolate instruction-tuning effects.

- Model (Gemma3-270M)
- Data

In [1]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.3 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
import transformers
import trl
import datasets

# Model


In [ ]:
MODEL_NAME = "google/gemma-3-270m-it"

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[INFO] Using device: {device}")

[INFO] Using device: cuda


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype='auto',
    device_map="auto",
    attn_implementation="eager"
)

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


print(f"[INFO] Model on devices: {model.device}")
print(f"[INFO] Model using dtypes: {model.dtype}")

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

[INFO] Model on devices: cuda:0
[INFO] Model using dtypes: torch.bfloat16


In [ ]:


# Model requires numbers (tokens) as input
# turn strings to tokens via a tokenizer

# model("Hello my name is dimeji")

In [ ]:
tokenizer("Hello my name is dimeji.", return_tensors='pt')

{'input_ids': tensor([[     2,   9259,   1041,   1463,    563,  89904,   5573, 236761]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
outputs = model((tokenizer("Hello my name is dimeji.", return_tensors='pt')["input_ids"]).to('cuda'))

outputs.keys()

odict_keys(['logits', 'past_key_values'])

In [ ]:
model

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

### Try the model with a pipeline

In [ ]:
from transformers import pipeline


pipe = pipeline("text-generation",
                model = MODEL_NAME,
                token = tokenizer)

input_text = "Hi my name is Dimeji"

input_prompt = pipe.tokenizer.apply_chat_template(input_text,
                                                  tokenize=False,
                                                  add_generation_prompt=True)

input_prompt

# 2. Dataset - HotpotQA (distractor config)
**Hop-1 distillation, defined**
For each Hotpotqa the teacher (GPT-4o) will give use a strucured decomposition of it first reasoning step in YAML (`thought`, `action`, `target_entity`). the student (Gamma-3-270M) will then learn to output this same reasoning steps given the question. The student is not responsible for retrieval or producing the final answer.

## 2.1 Schema exploration
**What:** Load a tiny slice of HotpotQA to inspect its schema before committing to a full pull.

**Why this config - `distractor` (not `fullwiki`):**
The student model is being trained to *decompose* a question into a first-hop sub-question, not to *retrieve* paragraphs from Wikipedia. `fullwiki` exits for retrieval-style setups; `distractor` ships each question with ~10 candidate paragraphs (gold + distractors), which is more than enough - and in fact more than the student will see at inference, since decomposition operates on the question alone.  

**Why `train[:50]`:**
Schema is identical at 50 rows or 90k. A 50-row slice is for inspection; the full pull happens later when we tokenize for training.

**Runs on:** `LOCAL OK` - pure inspection, no model forward pass.

In [3]:
from datasets import load_dataset
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split='train[:50]')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

In [6]:
print(ds)

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 50
})


In [10]:
#filter the dataset to only show rows where the type is 'comparison'
comparison_ds = ds.filter(lambda row: row['type'] == 'comparison')

view_ds = comparison_ds.select_columns(['question', 'supporting_facts', 'answer'])
print(view_ds[0])


Filter:   0%|          | 0/50 [00:00<?, ? examples/s]

{'question': "Which magazine was started first Arthur's Magazine or First for Women?", 'supporting_facts': {'title': ["Arthur's Magazine", 'First for Women'], 'sent_id': [0, 0]}, 'answer': "Arthur's Magazine"}


In [3]:
view_ds['question']

Column(["Which magazine was started first Arthur's Magazine or First for Women?", 'Which tennis player won more Grand Slam titles, Henri Leconte or Jonathan Stark?', 'Which band was founded first, Hole, the rock band that Courtney Love was a frontwoman of, or The Wolfhounds?', 'Were Pavel Urysohn and Leonid Levin known for the same type of work?', 'Are both The New Pornographers and Kings of Leon American rock bands?'])

In [7]:
import pandas as pd
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 30) 


bridge_ds = ds.filter(lambda row: row['type'] == 'bridge')
view_bridge_ds = bridge_ds.select_columns(['question', 'supporting_facts', 'answer', 'context']).to_pandas()
view_bridge_ds.head()

Filter:   0%|          | 0/50 [00:00<?, ? examples/s]

,question,supporting_facts,answer,context
0,The Oberoi family is part of a hotel company t...,"{'title': ['Oberoi family', 'The Oberoi Group'...",Delhi,"{'title': ['Ritz-Carlton Jakarta', 'Oberoi fam..."
1,Musician and satirist Allie Goertz wrote a son...,"{'title': ['Allie Goertz', 'Allie Goertz', 'Al...",President Richard Nixon,"{'title': ['Lisa Simpson', 'Marge Simpson', 'B..."
2,What nationality was James Henry Miller's wife?,"{'title': ['Peggy Seeger', 'Peggy Seeger', 'Ew...",American,"{'title': ['Moloch: or, This Gentile World', '..."
3,Cadmium Chloride is slightly soluble in this c...,"{'title': ['Cadmium chloride', 'Ethanol'], 'se...",alcohol,"{'title': ['Cadmium chloride', 'Water blue', '..."
4,Which genus of moth in the world's seventh-lar...,"{'title': ['Indogrammodes', 'Indogrammodes', '...",Crambidae,"{'title': ['India', 'List of companies of Indi..."


In [8]:
print(view_bridge_ds['question'][0])
print(view_bridge_ds['supporting_facts'][0])
print(view_bridge_ds['context'][0]['title'])
print(view_bridge_ds['answer'][0])

The Oberoi family is part of a hotel company that has a head office in what city?
{'title': array(['Oberoi family', 'The Oberoi Group'], dtype=object), 'sent_id': array([0, 0], dtype=int32)}
['Ritz-Carlton Jakarta' 'Oberoi family' 'Ishqbaaaz' 'Hotel Tallcorn'
 'Mohan Singh Oberoi' 'Hotel Bond' 'The Oberoi Group'
 'Future Fibre Technologies' '289th Military Police Company'
 'Glennwanis Hotel']
Delhi


In [ ]:
comparison_ds[0]['context']

{'title': ['Radio City (Indian radio station)',
  'History of Albanian football',
  'Echosmith',
  "Women's colleges in the Southern United States",
  'First Arthur County Courthouse and Jail',
  "Arthur's Magazine",
  '2014–15 Ukrainian Hockey Championship',
  'First for Women',
  'Freeway Complex Fire',
  'William Rast'],
 'sentences': [["Radio City is India's first private FM radio station and was started on 3 July 2001.",
   ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).',
   ' It plays Hindi, English and regional songs.',
   ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.',
   ' Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.',
   ' The Radio station c